# Compare greedy, hybrid, and dynamic programming

This notebook evaluates the complete hybrid solver. The classifier predicts whether ratio-greedy will fail. If its predicted failure probability exceeds `0.5`, the hybrid runs exact dynamic programming; otherwise, it accepts greedy.

The model is trained on the training split and evaluated on the validation split. The test split remains untouched for the later distribution-shift stage.

In [1]:
import ast
from time import perf_counter

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from knapsack_ml_experiment import (
    FEATURE_COLUMNS,
    Item,
    extract_hybrid_features,
    solve_dynamic_programming,
    solve_greedy,
    solve_hybrid,
    split_dataset_by_seed,
)

## Load the labeled instances

The saved optimal values are used only to score the returned solutions. They are not classifier inputs.

In [2]:
independent = pd.read_csv("../artifacts/independent_instances.csv")
other_families = pd.read_csv("../artifacts/other_family_instances.csv")
dataset = pd.concat([independent, other_families], ignore_index=True)

dataset["weights"] = dataset["weights"].apply(ast.literal_eval)
dataset["values"] = dataset["values"].apply(ast.literal_eval)


def items_from_row(row):
    return [
        Item(f"item_{index}", weight, value)
        for index, (weight, value) in enumerate(zip(row.weights, row.values))
    ]


feature_rows = [
    extract_hybrid_features(items_from_row(row), row.capacity).iloc[0].to_dict()
    for row in dataset.itertuples(index=False)
]
features = pd.DataFrame(feature_rows, columns=FEATURE_COLUMNS)
comparison_data = dataset.copy()
for column in FEATURE_COLUMNS:
    comparison_data[column] = features[column]

train, validation, _test = split_dataset_by_seed(
    comparison_data,
    random_seed=0,
)

{
    "training_rows": len(train),
    "validation_rows": len(validation),
    "held_out_test_rows": len(_test),
}

{'training_rows': 2800, 'validation_rows': 600, 'held_out_test_rows': 600}

## Train the same logistic-regression baseline

Only raw-instance features are supplied to the model. In particular, `failure`, `optimal_value`, and `relative_gap` are excluded to prevent answer leakage. The `liblinear` optimizer is used because this is a binary classification problem and it avoids the numerical warnings produced by the default optimizer on this dataset.

In [3]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, solver="liblinear"),
)
model.fit(train[list(FEATURE_COLUMNS)], train["failure"].astype(int))

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(max_iter=1000, solver='liblinear'))])

## Run all three strategies

Each strategy is timed end to end on the same validation instances. The hybrid timing includes feature extraction, probability prediction, and whichever solver it selects.

In [4]:
THRESHOLD = 0.5


def run_strategy(name, rows):
    records = []
    start = perf_counter()

    for row in rows.itertuples(index=False):
        items = items_from_row(row)

        if name == "always_greedy":
            solution = solve_greedy(items, row.capacity)
            used_dp = False
        elif name == "hybrid":
            result = solve_hybrid(items, row.capacity, model, THRESHOLD)
            solution = result.solution
            used_dp = result.solver_used == "dynamic_programming"
        elif name == "always_dp":
            solution = solve_dynamic_programming(items, row.capacity)
            used_dp = True
        else:
            raise ValueError(f"unknown strategy: {name}")

        relative_gap = (
            (row.optimal_value - solution.total_value) / row.optimal_value
            if row.optimal_value > 0
            else 0.0
        )
        records.append(
            {
                "family": row.family,
                "actual_greedy_failure": bool(row.failure),
                "solution_value": solution.total_value,
                "optimal_value": row.optimal_value,
                "relative_gap": relative_gap,
                "used_dp": used_dp,
            }
        )

    elapsed_seconds = perf_counter() - start
    return pd.DataFrame(records), elapsed_seconds


strategy_runs = {
    name: run_strategy(name, validation)
    for name in ("always_greedy", "hybrid", "always_dp")
}

## Summarize solution quality and computational cost

A hybrid false negative is an instance where greedy truly fails but the hybrid does not send it to DP. This is the most costly classifier mistake because the returned solution can be suboptimal.

In [5]:
summary_rows = []

for name, (results, elapsed_seconds) in strategy_runs.items():
    false_negatives = (
        results["actual_greedy_failure"] & ~results["used_dp"]
    ).sum()
    summary_rows.append(
        {
            "strategy": name,
            "optimal_solution_rate": (
                results["solution_value"] == results["optimal_value"]
            ).mean(),
            "mean_relative_gap": results["relative_gap"].mean(),
            "maximum_relative_gap": results["relative_gap"].max(),
            "percentage_sent_to_dp": 100 * results["used_dp"].mean(),
            "runtime_seconds": elapsed_seconds,
            "false_negatives": int(false_negatives),
        }
    )

comparison = pd.DataFrame(summary_rows).set_index("strategy")
comparison.round(4)

,optimal_solution_rate,mean_relative_gap,maximum_relative_gap,percentage_sent_to_dp,runtime_seconds,false_negatives
strategy,,,,,,
always_greedy,0.2600,0.0200,0.0941,0.0000,0.0082,444
hybrid,0.9333,0.0008,0.0431,85.1667,0.6054,40
always_dp,1.0000,0.0000,0.0000,100.0000,0.3956,0


## Interpret the comparison

At threshold `0.5`, the hybrid returns an optimal solution on 93.33% of validation instances while sending 85.17% to DP. This is much better than always-greedy's 26% optimal-solution rate, but it still misses 40 actual greedy failures. Its mean relative gap is small (`0.0008`), although its worst missed case loses 4.31% of the optimal value.

In this implementation, the hybrid is slower than always-DP despite making fewer DP calls. It predicts one row at a time with pandas and scikit-learn, so classifier and feature-construction overhead outweigh the saved DP work on these small 20-item instances. This is an experimental result, not evidence that hybrid routing can never improve runtime.

Do not read runtime too precisely from one notebook run. It depends on the machine and background activity. The quality metrics and DP-usage rate are deterministic for this fixed split, model, and threshold.